# CWRU Spectrogram Training Dataset Builder

이 노트북은 CWRU `.mat` 진동 신호를 **AI 학습용 spectrogram 이미지 데이터셋**으로 변환하는 파일이다.

핵심 기능:

- sliding window로 한 파일에서 여러 spectrogram 생성
- `hop_length` 조절로 overlap 정도 변경 가능
- `max_segments_per_file=-1`이면 가능한 모든 sliding window 사용
- train/val/test split 자동 생성
- `dataset_images/class_name/*.png` 형태 저장
- `metadata.csv`, `label_map.json`, `split.csv` 저장
- 파일명 끝 `_1`, `_2`는 label이 아니라 load condition으로 저장

In [1]:
# 필요한 라이브러리
# 처음 실행하는 환경에서 에러가 나면 아래 주석을 해제해서 설치하면 된다.
# !pip install numpy scipy pandas matplotlib pillow scikit-learn tqdm

from pathlib import Path
import re
import json
import random
import shutil
import numpy as np
import pandas as pd
from PIL import Image
from scipy.io import loadmat
from scipy import signal
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

## 1. 경로와 sliding window 설정

In [22]:
# =========================
# 사용자가 수정할 설정
# =========================

# .mat 파일들이 들어있는 폴더
MAT_DIR = Path("./mat_files")

# 결과 저장 폴더
OUTPUT_DIR = Path("./cwru_spectrogram_training_dataset")

# 사용할 센서 채널
CHANNEL = "DE"  # "DE" 또는 "FE"

# 샘플링 주파수
# 네 파일명이 48K 계열이면 48000으로 바꿔야 한다.
FS = 48000

# Sliding window 설정
SEGMENT_LENGTH = 4096

# hop_length가 작을수록 overlap이 커지고 이미지 수가 많아진다.
# 4096: overlap 없음
# 2048: 50% overlap
# 1024: 75% overlap
# 512 : 87.5% overlap
HOP_LENGTH = 3072

# 파일당 최대 segment 수
# -1이면 가능한 모든 sliding window를 저장한다.
MAX_SEGMENTS_PER_FILE = -1

# STFT 설정
NPERSEG = 256
NOVERLAP = 128

# 학습용 이미지 크기
IMAGE_SIZE = 224

# True: 1채널 흑백 PNG
# False: colormap RGB PNG
SAVE_GRAYSCALE = True

# train/val/test split 비율
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# 랜덤 seed
SEED = 42

# 기존 OUTPUT_DIR 삭제 후 새로 생성할지 여부
RESET_OUTPUT_DIR = True

In [23]:
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-8

if RESET_OUTPUT_DIR and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

DATASET_DIR = OUTPUT_DIR / "dataset_images"
SPLIT_DIR = OUTPUT_DIR / "split_dataset"

DATASET_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

## 2. Label 규칙

In [24]:
def label_from_filename(path: Path) -> str:
    # 마지막 _1, _2 같은 부하 번호를 제거해서 label을 만든다.
    # Fault_BALL_007_1.mat -> Fault_BALL_007
    # Normal_2.mat -> Normal
    return re.sub(r"_\d+$", "", path.stem)


def load_from_filename(path: Path) -> str:
    # 파일명 끝의 _1, _2를 부하 조건으로 추출한다.
    m = re.search(r"_(\d+)$", path.stem)
    return m.group(1) if m else "unknown"


def fault_type_from_label(label: str) -> str:
    # Fault_BALL_007 -> BALL
    # Fault_IR_014 -> IR
    # Normal -> Normal
    if label == "Normal":
        return "Normal"
    parts = label.split("_")
    return parts[1] if len(parts) >= 3 else "unknown"


def fault_size_from_label(label: str) -> str:
    # Fault_BALL_007 -> 007
    # Fault_IR_014 -> 014
    # Normal -> 000
    if label == "Normal":
        return "000"
    parts = label.split("_")
    return parts[2] if len(parts) >= 3 else "unknown"

## 3. `.mat` 신호 로드 함수

In [25]:
def find_signal_key(mat_dict: dict, channel: str = "DE") -> str:
    # CWRU .mat 내부에서 진동 신호 key를 자동으로 찾는다.
    # 우선 *_DE_time 또는 *_FE_time을 찾고,
    # 없으면 *_time 중 가장 긴 1D 신호를 사용한다.
    channel = channel.upper()
    candidates = []

    for k, v in mat_dict.items():
        if k.startswith("__"):
            continue
        if f"_{channel}_time" in k:
            arr = np.asarray(v).squeeze()
            if arr.ndim == 1 and arr.size > 100:
                candidates.append((k, arr.size))

    if not candidates:
        for k, v in mat_dict.items():
            if k.startswith("__"):
                continue
            if "_time" in k:
                arr = np.asarray(v).squeeze()
                if arr.ndim == 1 and arr.size > 100:
                    candidates.append((k, arr.size))

    if not candidates:
        raise ValueError("유효한 vibration time signal을 찾지 못했다.")

    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[0][0]


def load_vibration_signal(mat_path: Path, channel: str = "DE"):
    mat = loadmat(mat_path)
    signal_key = find_signal_key(mat, channel)
    x = np.asarray(mat[signal_key]).squeeze().astype(np.float64)

    # 평균 제거 + 표준편차 정규화
    x = x - np.mean(x)
    std = np.std(x)
    if std > 1e-12:
        x = x / std

    return x, signal_key

## 4. Sliding window + Spectrogram 함수

In [26]:
def generate_sliding_segments(x, segment_length, hop_length, max_segments_per_file=-1):
    # 긴 1D vibration signal을 sliding window로 자른다.
    # max_segments_per_file=-1이면 전체 window 사용
    if len(x) < segment_length:
        return []

    starts = np.arange(0, len(x) - segment_length + 1, hop_length)

    if max_segments_per_file is not None and max_segments_per_file > 0:
        if len(starts) > max_segments_per_file:
            idx = np.linspace(0, len(starts) - 1, max_segments_per_file).round().astype(int)
            starts = starts[idx]

    return [(int(s), x[int(s):int(s) + segment_length]) for s in starts]


def compute_spectrogram_db(seg, fs, nperseg, noverlap):
    f, t, S = signal.spectrogram(
        seg,
        fs=fs,
        window="hann",
        nperseg=nperseg,
        noverlap=noverlap,
        detrend=False,
        scaling="density",
        mode="magnitude",
    )
    S_db = 20 * np.log10(S + 1e-12)
    return f, t, S_db


def spectrogram_to_uint8(S_db, p_low=1, p_high=99):
    # dB spectrogram을 0~255 이미지로 변환한다.
    # percentile clipping으로 극단값 영향을 줄인다.
    vmin, vmax = np.percentile(S_db, [p_low, p_high])
    if vmax <= vmin:
        vmax = vmin + 1e-6

    img = np.clip((S_db - vmin) / (vmax - vmin), 0, 1)
    img = (img * 255).astype(np.uint8)

    # 낮은 주파수가 아래에 보이도록 뒤집기
    img = np.flipud(img)
    return img


def save_training_image(S_db, save_path: Path, image_size=224, grayscale=True):
    img = spectrogram_to_uint8(S_db)

    if grayscale:
        pil = Image.fromarray(img, mode="L")
        pil = pil.resize((image_size, image_size), Image.Resampling.BICUBIC)
    else:
        import matplotlib.cm as cm
        cmap = cm.get_cmap("viridis")
        rgba = cmap(img / 255.0)
        rgb = (rgba[:, :, :3] * 255).astype(np.uint8)
        pil = Image.fromarray(rgb, mode="RGB")
        pil = pil.resize((image_size, image_size), Image.Resampling.BICUBIC)

    save_path.parent.mkdir(parents=True, exist_ok=True)
    pil.save(save_path)

## 5. 예상 이미지 수 미리 계산

In [27]:
mat_files = sorted(MAT_DIR.glob("*.mat"))

if len(mat_files) == 0:
    raise FileNotFoundError(f"{MAT_DIR} 폴더에서 .mat 파일을 찾지 못했다.")

preview_rows = []

for mat_path in mat_files:
    x, signal_key = load_vibration_signal(mat_path, CHANNEL)
    starts = np.arange(0, len(x) - SEGMENT_LENGTH + 1, HOP_LENGTH)

    if MAX_SEGMENTS_PER_FILE > 0 and len(starts) > MAX_SEGMENTS_PER_FILE:
        n_selected = MAX_SEGMENTS_PER_FILE
    else:
        n_selected = len(starts)

    preview_rows.append({
        "source_file": mat_path.name,
        "label": label_from_filename(mat_path),
        "load_condition": load_from_filename(mat_path),
        "signal_key": signal_key,
        "signal_length": len(x),
        "possible_segments": len(starts),
        "selected_segments": n_selected,
    })

preview_df = pd.DataFrame(preview_rows)
print("총 예상 이미지 수:", int(preview_df["selected_segments"].sum()))
preview_df

총 예상 이미지 수: 3154


,source_file,label,load_condition,signal_key,signal_length,possible_segments,selected_segments
0,Fault_BALL_007_1.mat,Fault_BALL_007,1,X123_DE_time,487384,158,158
1,Fault_BALL_007_2.mat,Fault_BALL_007,2,X124_DE_time,486804,158,158
2,Fault_BALL_014_1.mat,Fault_BALL_014,1,X190_DE_time,486224,157,157
3,Fault_BALL_014_2.mat,Fault_BALL_014,2,X191_DE_time,487384,158,158
4,Fault_BALL_021_1.mat,Fault_BALL_021,1,X227_DE_time,486804,158,158
5,Fault_BALL_021_2.mat,Fault_BALL_021,2,X228_DE_time,487384,158,158
6,Fault_IR_007_1.mat,Fault_IR_007,1,X110_DE_time,486224,157,157
7,Fault_IR_007_2.mat,Fault_IR_007,2,X111_DE_time,485643,157,157
8,Fault_IR_014_1.mat,Fault_IR_014,1,X217_DE_time,489125,158,158
9,Fault_IR_014_2.mat,Fault_IR_014,2,X176_DE_time,487964,158,158


## 6. 학습용 spectrogram 이미지 생성

In [28]:
metadata = []

labels = sorted({label_from_filename(p) for p in mat_files})
label_to_id = {label: i for i, label in enumerate(labels)}

with open(OUTPUT_DIR / "label_map.json", "w", encoding="utf-8") as f:
    json.dump(label_to_id, f, ensure_ascii=False, indent=2)

for mat_path in tqdm(mat_files):
    label = label_from_filename(mat_path)
    label_id = label_to_id[label]
    load_condition = load_from_filename(mat_path)
    fault_type = fault_type_from_label(label)
    fault_size = fault_size_from_label(label)

    x, signal_key = load_vibration_signal(mat_path, CHANNEL)
    segments = generate_sliding_segments(
        x,
        segment_length=SEGMENT_LENGTH,
        hop_length=HOP_LENGTH,
        max_segments_per_file=MAX_SEGMENTS_PER_FILE,
    )

    for seg_idx, (start_sample, seg) in enumerate(segments):
        f, t, S_db = compute_spectrogram_db(seg, FS, NPERSEG, NOVERLAP)

        file_name = f"{mat_path.stem}__seg{seg_idx:05d}__start{start_sample}.png"
        save_path = DATASET_DIR / label / file_name

        save_training_image(
            S_db,
            save_path,
            image_size=IMAGE_SIZE,
            grayscale=SAVE_GRAYSCALE,
        )

        metadata.append({
            "image_path": str(save_path.relative_to(OUTPUT_DIR)).replace("\\", "/"),
            "label": label,
            "label_id": label_id,
            "load_condition": load_condition,
            "fault_type": fault_type,
            "fault_size": fault_size,
            "source_file": mat_path.name,
            "signal_key": signal_key,
            "channel": CHANNEL,
            "fs": FS,
            "segment_index": seg_idx,
            "start_sample": start_sample,
            "segment_length": SEGMENT_LENGTH,
            "hop_length": HOP_LENGTH,
            "nperseg": NPERSEG,
            "noverlap": NOVERLAP,
            "image_size": IMAGE_SIZE,
            "grayscale": SAVE_GRAYSCALE,
        })

metadata_df = pd.DataFrame(metadata)
metadata_df.to_csv(OUTPUT_DIR / "metadata.csv", index=False, encoding="utf-8-sig")

print("생성 완료")
print("총 이미지 수:", len(metadata_df))
metadata_df.head()

  0%|          | 0/20 [00:00<?, ?it/s]

생성 완료
총 이미지 수: 3154


,image_path,label,label_id,load_condition,fault_type,fault_size,source_file,signal_key,channel,fs,segment_index,start_sample,segment_length,hop_length,nperseg,noverlap,image_size,grayscale
0,dataset_images/Fault_BALL_007/Fault_BALL_007_1...,Fault_BALL_007,0,1,BALL,007,Fault_BALL_007_1.mat,X123_DE_time,DE,48000,0,0,4096,3072,256,128,224,True
1,dataset_images/Fault_BALL_007/Fault_BALL_007_1...,Fault_BALL_007,0,1,BALL,007,Fault_BALL_007_1.mat,X123_DE_time,DE,48000,1,3072,4096,3072,256,128,224,True
2,dataset_images/Fault_BALL_007/Fault_BALL_007_1...,Fault_BALL_007,0,1,BALL,007,Fault_BALL_007_1.mat,X123_DE_time,DE,48000,2,6144,4096,3072,256,128,224,True
3,dataset_images/Fault_BALL_007/Fault_BALL_007_1...,Fault_BALL_007,0,1,BALL,007,Fault_BALL_007_1.mat,X123_DE_time,DE,48000,3,9216,4096,3072,256,128,224,True
4,dataset_images/Fault_BALL_007/Fault_BALL_007_1...,Fault_BALL_007,0,1,BALL,007,Fault_BALL_007_1.mat,X123_DE_time,DE,48000,4,12288,4096,3072,256,128,224,True


## 7. Class별 이미지 수 확인

In [29]:
class_count = metadata_df.groupby(["label", "label_id"]).size().reset_index(name="num_images")
class_count.to_csv(OUTPUT_DIR / "class_count.csv", index=False, encoding="utf-8-sig")
class_count

,label,label_id,num_images
0,Fault_BALL_007,0,316
1,Fault_BALL_014,1,315
2,Fault_BALL_021,2,316
3,Fault_IR_007,3,314
4,Fault_IR_014,4,316
5,Fault_IR_021,5,316
6,Fault_OR_007,6,316
7,Fault_OR_014,7,315
8,Fault_OR_021,8,316
9,Normal,9,314


## 8. Train / Validation / Test split 생성

In [30]:
# 이미지 단위 stratified split
# 주의: sliding window 특성상 같은 원본 파일에서 나온 이미지가 train/test에 섞일 수 있다.
# 처음 모델이 잘 도는지 확인하는 용도로는 괜찮지만,
# 최종 성능 주장에는 source_file 또는 load_condition 기준 split도 고려해야 한다.

train_df, temp_df = train_test_split(
    metadata_df,
    train_size=TRAIN_RATIO,
    stratify=metadata_df["label"],
    random_state=SEED,
)

relative_val_ratio = VAL_RATIO / (VAL_RATIO + TEST_RATIO)

val_df, test_df = train_test_split(
    temp_df,
    train_size=relative_val_ratio,
    stratify=temp_df["label"],
    random_state=SEED,
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

split_df = pd.concat([train_df, val_df, test_df], axis=0).reset_index(drop=True)
split_df.to_csv(OUTPUT_DIR / "split.csv", index=False, encoding="utf-8-sig")

split_df.groupby(["split", "label"]).size().unstack(fill_value=0)

label,Fault_BALL_007,Fault_BALL_014,Fault_BALL_021,Fault_IR_007,Fault_IR_014,Fault_IR_021,Fault_OR_007,Fault_OR_014,Fault_OR_021,Normal
split,,,,,,,,,,
test,48,47,47,47,48,47,48,47,48,47
train,221,221,221,220,221,221,221,220,221,220
val,47,47,48,47,47,48,47,48,47,47


## 9. split_dataset 폴더 생성

In [31]:
# split_dataset/train/class/*.png
# split_dataset/val/class/*.png
# split_dataset/test/class/*.png
# 구조로 복사한다.
# PyTorch ImageFolder에서 바로 읽을 수 있다.

for _, row in tqdm(split_df.iterrows(), total=len(split_df)):
    src = OUTPUT_DIR / row["image_path"]
    dst = SPLIT_DIR / row["split"] / row["label"] / Path(row["image_path"]).name
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

print("split_dataset 생성 완료")
print(SPLIT_DIR)

  0%|          | 0/3154 [00:00<?, ?it/s]

split_dataset 생성 완료
cwru_spectrogram_training_dataset\split_dataset


## 10. PyTorch ImageFolder로 불러오는 예시

In [32]:
# 이 셀은 실제 학습 코드에서 참고용으로 사용하면 된다.
# torchvision이 없으면 아래 주석을 해제해서 설치한다.
# !pip install torch torchvision

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # 흑백 1채널
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(
    root=OUTPUT_DIR / "split_dataset" / "train",
    transform=transform,
)

val_dataset = datasets.ImageFolder(
    root=OUTPUT_DIR / "split_dataset" / "val",
    transform=transform,
)

test_dataset = datasets.ImageFolder(
    root=OUTPUT_DIR / "split_dataset" / "test",
    transform=transform,
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(train_dataset.classes)
print(len(train_dataset), len(val_dataset), len(test_dataset))

['Fault_BALL_007', 'Fault_BALL_014', 'Fault_BALL_021', 'Fault_IR_007', 'Fault_IR_014', 'Fault_IR_021', 'Fault_OR_007', 'Fault_OR_014', 'Fault_OR_021', 'Normal']
2207 473 474


## 11. 중요한 실험 주의사항

### 이미지 수를 늘리는 방법

`HOP_LENGTH`를 줄이면 sliding window overlap이 커져서 이미지 수가 증가한다.

| `HOP_LENGTH` | 의미 |
|---:|---|
| `4096` | overlap 없음 |
| `2048` | 50% overlap |
| `1024` | 75% overlap |
| `512` | 87.5% overlap |

### 데이터 누수 주의

이미지 단위 split을 하면 같은 원본 `.mat` 파일에서 나온 인접 segment들이 train/test에 동시에 들어갈 수 있다.

논문/보고서에서 더 엄격하게 실험하려면 다음 split도 고려해야 한다.

- `load_condition=1`로 학습, `load_condition=2`로 테스트
- 특정 원본 파일을 통째로 test로 분리
- source_file 기준 Group split 사용